# Spectral Graph Theory - Exercises

Practice problems covering graph Laplacians, eigenvalues, spectral clustering, and graph signal processing.

In [ ]:
import numpy as np
from scipy import linalg
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigsh
import numpy.linalg as la

## Exercise 1: Graph Laplacian Construction

Given an adjacency matrix, construct:
1. The degree matrix D
2. The unnormalized Laplacian L = D - A
3. The symmetric normalized Laplacian L_sym = D^(-1/2) L D^(-1/2)
4. Verify that L is positive semi-definite

In [ ]:
# Your solution here
A = np.array([
    [0, 1, 1, 0, 0],
    [1, 0, 1, 1, 0],
    [1, 1, 0, 1, 1],
    [0, 1, 1, 0, 1],
    [0, 0, 1, 1, 0]
])

### Solution 1

In [ ]:
# Degree matrix
degrees = np.sum(A, axis=1)
D = np.diag(degrees)
print("Degree matrix D:")
print(D)

# Unnormalized Laplacian
L = D - A
print("\nUnnormalized Laplacian L:")
print(L)

# Symmetric normalized Laplacian
D_inv_sqrt = np.diag(1.0 / np.sqrt(degrees))
L_sym = D_inv_sqrt @ L @ D_inv_sqrt
print("\nSymmetric normalized Laplacian L_sym:")
print(np.round(L_sym, 4))

# Verify positive semi-definite (all eigenvalues >= 0)
eigenvalues = la.eigvalsh(L)
print("\nEigenvalues of L:", np.round(eigenvalues, 6))
print("All eigenvalues >= 0:", np.all(eigenvalues >= -1e-10))

## Exercise 2: Connected Components from Spectrum

The number of zero eigenvalues of the Laplacian equals the number of connected components.
1. Create a graph with 2 disconnected components
2. Compute the Laplacian eigenvalues
3. Verify the number of zero eigenvalues matches the components

In [ ]:
# Your solution here

### Solution 2

In [ ]:
# Graph with 2 disconnected components: triangle + edge
A_disconnected = np.array([
    [0, 1, 1, 0, 0],  # Component 1: nodes 0,1,2 (triangle)
    [1, 0, 1, 0, 0],
    [1, 1, 0, 0, 0],
    [0, 0, 0, 0, 1],  # Component 2: nodes 3,4 (edge)
    [0, 0, 0, 1, 0]
])

D_disc = np.diag(np.sum(A_disconnected, axis=1))
L_disc = D_disc - A_disconnected

eigenvalues = la.eigvalsh(L_disc)
print("Eigenvalues:", np.round(eigenvalues, 6))

num_zero = np.sum(np.abs(eigenvalues) < 1e-10)
print(f"Number of zero eigenvalues: {num_zero}")
print(f"Number of connected components: 2")
print(f"Match: {num_zero == 2}")

## Exercise 3: Spectral Clustering

Implement spectral clustering:
1. Compute the normalized Laplacian
2. Find the k smallest eigenvectors
3. Use k-means on the eigenvector embedding
4. Test on a graph with clear cluster structure

In [ ]:
# Your solution here

### Solution 3

In [ ]:
def spectral_clustering(A, k):
    """Spectral clustering using normalized Laplacian."""
    n = A.shape[0]
    
    # Degree and Laplacian
    degrees = np.sum(A, axis=1)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(degrees + 1e-10))
    L_sym = np.eye(n) - D_inv_sqrt @ A @ D_inv_sqrt
    
    # k smallest eigenvectors
    eigenvalues, eigenvectors = la.eigh(L_sym)
    embedding = eigenvectors[:, :k]
    
    # Normalize rows
    row_norms = la.norm(embedding, axis=1, keepdims=True)
    embedding_norm = embedding / (row_norms + 1e-10)
    
    # Simple k-means (2 clusters)
    # Initialize with first and last node
    centers = np.array([embedding_norm[0], embedding_norm[-1]])
    
    for _ in range(10):
        # Assign to nearest center
        dists = np.array([la.norm(embedding_norm - c, axis=1) for c in centers])
        labels = np.argmin(dists, axis=0)
        
        # Update centers
        for i in range(k):
            if np.sum(labels == i) > 0:
                centers[i] = np.mean(embedding_norm[labels == i], axis=0)
    
    return labels, eigenvalues[:k]

# Create graph with 2 clusters
# Cluster 1: nodes 0-4 (dense), Cluster 2: nodes 5-9 (dense)
# One edge connecting clusters
A_cluster = np.zeros((10, 10))
# Cluster 1 (complete subgraph)
A_cluster[:5, :5] = 1 - np.eye(5)
# Cluster 2 (complete subgraph)
A_cluster[5:, 5:] = 1 - np.eye(5)
# One bridge edge
A_cluster[4, 5] = A_cluster[5, 4] = 1

labels, eigs = spectral_clustering(A_cluster, k=2)
print("Cluster assignments:", labels)
print("Expected: [0,0,0,0,0,1,1,1,1,1] or [1,1,1,1,1,0,0,0,0,0]")
print("Smallest eigenvalues:", np.round(eigs, 4))

## Exercise 4: Graph Fourier Transform

Implement the Graph Fourier Transform:
1. Compute the eigenvectors of the Laplacian (Fourier basis)
2. Transform a graph signal to spectral domain
3. Apply a low-pass filter
4. Transform back to vertex domain

In [ ]:
# Your solution here

### Solution 4

In [ ]:
def graph_fourier_transform(L):
    """Compute GFT basis (Laplacian eigenvectors)."""
    eigenvalues, U = la.eigh(L)
    return eigenvalues, U

def gft(signal, U):
    """Graph Fourier Transform: vertex → spectral."""
    return U.T @ signal

def igft(spectral_signal, U):
    """Inverse GFT: spectral → vertex."""
    return U @ spectral_signal

def low_pass_filter(eigenvalues, cutoff):
    """Create low-pass filter in spectral domain."""
    return (eigenvalues < cutoff).astype(float)

# Example: Path graph
n = 10
A_path = np.diag(np.ones(n-1), 1) + np.diag(np.ones(n-1), -1)
D_path = np.diag(np.sum(A_path, axis=1))
L_path = D_path - A_path

# Get Fourier basis
eigenvalues, U = graph_fourier_transform(L_path)

# Create noisy signal (smooth + noise)
smooth_signal = np.sin(np.linspace(0, 2*np.pi, n))
noise = 0.5 * np.random.randn(n)
noisy_signal = smooth_signal + noise

# Apply low-pass filter
spectral = gft(noisy_signal, U)
h = low_pass_filter(eigenvalues, cutoff=1.0)
filtered_spectral = spectral * h
filtered_signal = igft(filtered_spectral, U)

print("Original signal:", np.round(smooth_signal, 2))
print("Noisy signal:", np.round(noisy_signal, 2))
print("Filtered signal:", np.round(filtered_signal, 2))
print("\nMSE before filtering:", np.mean((noisy_signal - smooth_signal)**2))
print("MSE after filtering:", np.mean((filtered_signal - smooth_signal)**2))

## Exercise 5: Cheeger Constant Approximation

The Fiedler vector (eigenvector for λ₂) approximates the minimum cut.
1. Compute the Fiedler vector
2. Use its sign to partition the graph
3. Compute the cut ratio

In [ ]:
# Your solution here

### Solution 5

In [ ]:
def fiedler_partition(A):
    """Partition graph using Fiedler vector."""
    D = np.diag(np.sum(A, axis=1))
    L = D - A
    
    eigenvalues, eigenvectors = la.eigh(L)
    
    # Fiedler vector (second eigenvector)
    fiedler = eigenvectors[:, 1]
    lambda2 = eigenvalues[1]
    
    # Partition by sign
    partition = (fiedler >= 0).astype(int)
    
    return partition, fiedler, lambda2

def compute_cut_ratio(A, partition):
    """Compute normalized cut ratio."""
    n = A.shape[0]
    S = np.where(partition == 0)[0]
    T = np.where(partition == 1)[0]
    
    # Cut edges
    cut = np.sum(A[np.ix_(S, T)])
    
    # Volumes
    vol_S = np.sum(A[S, :])
    vol_T = np.sum(A[T, :])
    
    # Normalized cut
    ncut = cut / vol_S + cut / vol_T if vol_S > 0 and vol_T > 0 else float('inf')
    
    return cut, ncut

# Use the cluster graph from Exercise 3
partition, fiedler, lambda2 = fiedler_partition(A_cluster)
cut, ncut = compute_cut_ratio(A_cluster, partition)

print("Fiedler vector:", np.round(fiedler, 3))
print("λ₂ (algebraic connectivity):", round(lambda2, 4))
print("Partition:", partition)
print(f"Cut edges: {cut}")
print(f"Normalized cut: {ncut:.4f}")

## Exercise 6: Spectral Embedding

Create a 2D embedding of a graph using the first 2 non-trivial eigenvectors.

In [ ]:
# Your solution here

### Solution 6

In [ ]:
def spectral_embedding(A, dim=2):
    """Embed graph vertices in dim-dimensional space."""
    D = np.diag(np.sum(A, axis=1))
    L = D - A
    
    eigenvalues, eigenvectors = la.eigh(L)
    
    # Skip first eigenvector (constant), take next 'dim'
    embedding = eigenvectors[:, 1:dim+1]
    
    return embedding, eigenvalues[1:dim+1]

# Create a cycle graph
n = 8
A_cycle = np.zeros((n, n))
for i in range(n):
    A_cycle[i, (i+1) % n] = 1
    A_cycle[(i+1) % n, i] = 1

embedding, eigs = spectral_embedding(A_cycle, dim=2)

print("2D Spectral Embedding of Cycle Graph:")
print("Node\t X\t Y")
for i in range(n):
    print(f"{i}\t{embedding[i,0]:.3f}\t{embedding[i,1]:.3f}")

print(f"\nEigenvalues used: {np.round(eigs, 4)}")
print("\nNote: Points should form a circle (cycle graph embeds to circle)")

## Exercise 7: Heat Diffusion on Graphs

Implement heat diffusion using the graph Laplacian:
$$\frac{\partial f}{\partial t} = -L f$$

Solution: $f(t) = e^{-tL} f(0)$

In [ ]:
# Your solution here

### Solution 7

In [ ]:
def heat_diffusion(L, f0, t):
    """Diffuse signal f0 for time t using heat equation."""
    eigenvalues, U = la.eigh(L)
    
    # Heat kernel in spectral domain
    heat_kernel = np.exp(-t * eigenvalues)
    
    # Apply: f(t) = U @ diag(e^{-t*λ}) @ U^T @ f0
    spectral_f0 = U.T @ f0
    spectral_ft = heat_kernel * spectral_f0
    ft = U @ spectral_ft
    
    return ft

# Path graph
n = 10
A_path = np.diag(np.ones(n-1), 1) + np.diag(np.ones(n-1), -1)
D_path = np.diag(np.sum(A_path, axis=1))
L_path = D_path - A_path

# Initial heat: concentrated at node 0
f0 = np.zeros(n)
f0[0] = 1.0

print("Heat diffusion on path graph:")
print(f"t=0: {np.round(f0, 3)}")
for t in [0.1, 0.5, 1.0, 2.0, 5.0]:
    ft = heat_diffusion(L_path, f0, t)
    print(f"t={t}: {np.round(ft, 3)}")

## Exercise 8: Polynomial Graph Filters

Implement a polynomial filter $g(L) = \sum_{k=0}^{K} \theta_k L^k$ (basis for ChebNet).

In [ ]:
# Your solution here

### Solution 8

In [ ]:
def polynomial_filter(L, signal, coeffs):
    """Apply polynomial filter to graph signal.
    
    g(L) = sum_k coeffs[k] * L^k
    """
    n = L.shape[0]
    result = np.zeros(n)
    
    L_power = np.eye(n)  # L^0 = I
    for k, theta in enumerate(coeffs):
        result += theta * (L_power @ signal)
        L_power = L_power @ L  # L^{k+1}
    
    return result

def chebyshev_filter(L, signal, coeffs):
    """Chebyshev polynomial filter (more stable).
    
    Uses recurrence: T_k(x) = 2x*T_{k-1}(x) - T_{k-2}(x)
    """
    # Rescale L to [-1, 1]: L_scaled = 2L/lambda_max - I
    lambda_max = la.eigvalsh(L)[-1]
    L_scaled = 2 * L / lambda_max - np.eye(L.shape[0])
    
    T_prev = signal  # T_0(L) * signal = signal
    T_curr = L_scaled @ signal  # T_1(L) * signal
    
    result = coeffs[0] * T_prev
    if len(coeffs) > 1:
        result += coeffs[1] * T_curr
    
    for k in range(2, len(coeffs)):
        T_next = 2 * L_scaled @ T_curr - T_prev
        result += coeffs[k] * T_next
        T_prev, T_curr = T_curr, T_next
    
    return result

# Test on path graph
signal = np.random.randn(n)

# Low-pass filter coefficients (approximate)
coeffs_poly = [1.0, -0.5, 0.1]  # 1 - 0.5*L + 0.1*L^2
coeffs_cheb = [0.5, -0.3, 0.1]

filtered_poly = polynomial_filter(L_path, signal, coeffs_poly)
filtered_cheb = chebyshev_filter(L_path, signal, coeffs_cheb)

print("Original signal:", np.round(signal, 3))
print("Polynomial filtered:", np.round(filtered_poly, 3))
print("Chebyshev filtered:", np.round(filtered_cheb, 3))

## Exercise 9: Random Walk and PageRank

Implement PageRank using the random walk Laplacian.

In [ ]:
# Your solution here

### Solution 9

In [ ]:
def pagerank(A, alpha=0.85, max_iter=100, tol=1e-6):
    """Compute PageRank using power iteration.
    
    PR = alpha * P^T @ PR + (1-alpha) * (1/n)
    where P = D^{-1} @ A is random walk transition matrix
    """
    n = A.shape[0]
    
    # Transition matrix
    degrees = np.sum(A, axis=1)
    D_inv = np.diag(1.0 / (degrees + 1e-10))
    P = D_inv @ A
    
    # Initialize
    pr = np.ones(n) / n
    
    # Power iteration
    for _ in range(max_iter):
        pr_new = alpha * (P.T @ pr) + (1 - alpha) / n
        
        if la.norm(pr_new - pr) < tol:
            break
        pr = pr_new
    
    return pr / np.sum(pr)  # Normalize

# Create a directed web-like graph
A_web = np.array([
    [0, 1, 1, 0, 0, 0],
    [0, 0, 1, 1, 0, 0],
    [1, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 1, 1],
    [0, 0, 1, 0, 0, 1],
    [1, 0, 0, 0, 0, 0]
])

pr = pagerank(A_web)
print("PageRank scores:")
for i, score in enumerate(pr):
    print(f"  Node {i}: {score:.4f}")

print(f"\nMost important node: {np.argmax(pr)}")

## Exercise 10: Spectral Sparsification

Approximate a complete graph with a sparse graph that preserves spectral properties.

In [ ]:
# Your solution here

### Solution 10

In [ ]:
def effective_resistance(L):
    """Compute effective resistance for all edges."""
    n = L.shape[0]
    
    # Pseudoinverse of Laplacian
    eigenvalues, U = la.eigh(L)
    # Replace zero eigenvalue with small value to avoid division
    eigenvalues[eigenvalues < 1e-10] = 1e-10
    L_pinv = U @ np.diag(1.0 / eigenvalues) @ U.T
    
    # Effective resistance R_ij = L_pinv[i,i] + L_pinv[j,j] - 2*L_pinv[i,j]
    R = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            R[i,j] = L_pinv[i,i] + L_pinv[j,j] - 2*L_pinv[i,j]
            R[j,i] = R[i,j]
    
    return R

def spectral_sparsify(A, num_edges):
    """Sparsify graph by sampling edges proportional to effective resistance."""
    n = A.shape[0]
    D = np.diag(np.sum(A, axis=1))
    L = D - A
    
    R = effective_resistance(L)
    
    # Get all edges
    edges = []
    probs = []
    for i in range(n):
        for j in range(i+1, n):
            if A[i,j] > 0:
                edges.append((i, j))
                probs.append(R[i,j])
    
    probs = np.array(probs)
    probs /= probs.sum()
    
    # Sample edges
    sampled_idx = np.random.choice(len(edges), size=min(num_edges, len(edges)), 
                                    replace=False, p=probs)
    
    # Build sparse graph
    A_sparse = np.zeros((n, n))
    for idx in sampled_idx:
        i, j = edges[idx]
        weight = 1.0 / (num_edges * probs[idx])  # Importance weight
        A_sparse[i, j] = weight
        A_sparse[j, i] = weight
    
    return A_sparse

# Complete graph
n = 6
A_complete = np.ones((n, n)) - np.eye(n)

# Sparsify to ~n*log(n) edges
A_sparse = spectral_sparsify(A_complete, num_edges=int(n * np.log(n) * 2))

print(f"Original edges: {int(np.sum(A_complete)/2)}")
print(f"Sparse edges: {int(np.sum(A_sparse > 0)/2)}")

# Compare eigenvalues
D_orig = np.diag(np.sum(A_complete, axis=1))
L_orig = D_orig - A_complete
D_sparse = np.diag(np.sum(A_sparse, axis=1))
L_sparse = D_sparse - A_sparse

eig_orig = la.eigvalsh(L_orig)
eig_sparse = la.eigvalsh(L_sparse)

print("\nOriginal Laplacian eigenvalues:", np.round(eig_orig, 3))
print("Sparse Laplacian eigenvalues:", np.round(eig_sparse, 3))